## NB_OTT_EmbeddingEvaluation

### Objetivo:
Evaluar experimentalmente si los embeddings generados capturan información semántica útil para relacionar tickets correspondientes al mismo incidente.
Este notebook NO forma parte del pipeline operacional.
Aquí sí se utiliza incident_id porque constituye el ground truth del dataset sintético y permite evaluar la calidad de la representación semántica.
incident_id NO fue utilizado para generar los embeddings.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import functions as F

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

In [ ]:
# ============================================================
# 1. Preparación del dataset experimental
#
# En esta parte vamos a combinar:
#
# Silver.Tickets
#       +
# Silver.TicketEmbeddings
#
# De esta forma vamos a disponer simultáneamente de:
# - contenido del ticket
# - ground truth (incident_id)
# - embedding
# ============================================================

df_tickets = (
    spark.table("Silver.Tickets")
    .select(
        "ticket_id",
        "title",
        "description",
        "incident_id",
        "is_isolated"
    )
)

df_embeddings = (
    spark.table(
        "Silver.TicketEmbeddings"
    )
    .select(
        "ticket_id",
        "embedding",
        "embedding_model",
        "embedding_dimension"
    )
)

df_validation = (
    df_tickets
    .join(
        df_embeddings,
        on="ticket_id",
        how="inner"
    )
)

print(
    "Tickets available:",
    df_validation.count()
)

In [ ]:
# ============================================================
# 2. Selección del conjunto evaluable
#
# Para esta evaluación se estudia específicamente si tickets pertenecientes a un mismo incidente presentan mayor similitud que tickets pertenecientes a incidentes distintos.
#
# Los tickets aislados no tienen un incidente recurrente contra el que realizar esta comparación, por lo que quedan excluidos de este análisis.
# ============================================================

df_validation = (
    df_validation
    .filter(
        F.col("incident_id").isNotNull()
    )
    .filter(
        ~F.coalesce(
            F.col("is_isolated"),
            F.lit(False)
        )
    )
)

evaluation_count = (
    df_validation.count()
)

print(
    "Tickets used for embedding evaluation:",
    evaluation_count
)

assert evaluation_count > 1

In [ ]:
# ============================================================
# 3. Conversión a memoria mediante toPandas()
#
# Ya que el volumen de los datos no es demasiado alto (1000 tickets), vamos a usar pandas para trabajar con ellos.
#
# Esto no sería adecuado para millones de tickets, ya que la complejidad computacional es alta. Para hacer esto en producción se utilizarían técnicas como approximate nearest neighbours, particionado o candidate generation.
# ============================================================

pdf = df_validation.toPandas()

X = np.vstack(
    pdf["embedding"]
    .apply(np.array)
    .values
)

print("Tickets:", len(pdf))
print("Embedding matrix:", X.shape)

In [ ]:
# ============================================================
# 4. Validación de la normalización de los embeddings
#
# Los embeddings fueron generados con normalize_embeddings=True. Por tanto, la normalización L2 de cada vector debería ser aproximadamente 1. En esta parte comprobamos que es así.
# ============================================================

norms = np.linalg.norm(
    X,
    axis=1
)

print(
    "Mean vector norm:",
    norms.mean()
)

print(
    "Min vector norm:",
    norms.min()
)

print(
    "Max vector norm:",
    norms.max()
)

assert np.allclose(
    norms,
    1.0,
    atol=1e-3
), "Embeddings are not normalized as expected."

In [ ]:
# ============================================================
# 5. Cálculo de similitud semántica
#
# Al estar los embeddings normalizados:
#
#     cosine_similarity(a, b) = a · b
#
# Podemos calcular la matriz completa de similitudes mediante producto matricial. La diagonal siempre será aproximadamente 1, porque cada ticket se compara consigo mismo. Esto permite obtener de forma eficiente una matriz de similitud que cuantifica el grado de proximidad semántica entre cada pareja de tickets.
# ============================================================

similarity_matrix = X @ X.T

# Solo necesitamos una mitad de la matriz:
# comparar A-B es equivalente a comparar B-A.
rows, cols = np.triu_indices(
    len(pdf),
    k=1
)

similarities = (
    similarity_matrix[
        rows,
        cols
    ]
)

incident_ids = (
    pdf["incident_id"]
    .astype(str)
    .to_numpy()
)

same_incident = (
    incident_ids[rows]
    ==
    incident_ids[cols]
)

print(
    "Total ticket pairs:",
    len(similarities)
)

print(
    "Same-incident pairs:",
    int(same_incident.sum())
)

print(
    "Different-incident pairs:",
    int((~same_incident).sum())
)

In [ ]:
# ============================================================
# 6. Dataset pairwise
#
# Cada fila representa una pareja de tickets. De esta manera:
#
# same_incident=True
#     ambos pertenecen al mismo incidente real.
#
# same_incident=False
#     pertenecen a incidentes distintos.
# ============================================================

pairs = pd.DataFrame({
    "similarity":
        similarities,

    "same_incident":
        same_incident
})

pairs["group"] = np.where(
    pairs["same_incident"],
    "Same incident",
    "Different incident"
)

display(
    pairs.head()
)

print(
    pairs[
        "same_incident"
    ].value_counts()
)

In [ ]:
# ============================================================
# 7. Comparación estadística de las distribuciones
#
# Si los embeddings contienen información útil esperamos:
#
# media(same incident)
#       >
# media(different incident)
#
# aunque las distribuciones pueden presentar solapamiento.
# ============================================================

summary = (
    pairs
    .groupby(
        "group"
    )["similarity"]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max"
    ])
)

display(summary)

In [ ]:
mean_same = (
    pairs.loc[
        pairs["same_incident"],
        "similarity"
    ].mean()
)

mean_different = (
    pairs.loc[
        ~pairs["same_incident"],
        "similarity"
    ].mean()
)

print(
    "Mean same incident:",
    round(mean_same, 4)
)

print(
    "Mean different incident:",
    round(mean_different, 4)
)

print(
    "Difference:",
    round(
        mean_same - mean_different,
        4
    )
)

In [ ]:
# ============================================================
# 8. Distribución de similitud
#
# Esta visualización permite observar:
#
# - si los tickets del mismo incidente tienden a tener mayor
#   similitud;
# - cuánto solapamiento existe entre ambos conjuntos.
#
# El solapamiento será importante posteriormente para explicar
# los falsos positivos del clustering.
# ============================================================

same = (
    pairs.loc[
        pairs["same_incident"],
        "similarity"
    ]
)

different = (
    pairs.loc[
        ~pairs["same_incident"],
        "similarity"
    ]
)

plt.figure(
    figsize=(10, 6)
)

plt.hist(
    different,
    bins=50,
    alpha=0.6,
    density=True,
    label="Different incident"
)

plt.hist(
    same,
    bins=50,
    alpha=0.6,
    density=True,
    label="Same incident"
)

plt.xlabel(
    "Cosine similarity"
)

plt.ylabel(
    "Density"
)

plt.title(
    "Embedding similarity distribution"
)

plt.legend()

plt.show()

In [ ]:
# ============================================================
# 9. Percentiles
#
# Los percentiles permiten estudiar el solapamiento de forma más detallada que únicamente comparando medias.
# ============================================================

percentiles = [
    0.05,
    0.10,
    0.25,
    0.50,
    0.75,
    0.90,
    0.95
]

print("SAME INCIDENT")

print(
    same.describe(
        percentiles=percentiles
    )
)

print(
    "\nDIFFERENT INCIDENT"
)

print(
    different.describe(
        percentiles=percentiles
    )
)

In [ ]:
# ============================================================
# 10. Hard negatives
#
# Son pares pertenecientes a incidentes DIFERENTES pero con similitud semántica muy elevada.
#
# Son especialmente relevantes porque representan candidatos naturales a falsos positivos del clustering posterior.
# ============================================================

HIGH_SIMILARITY = 0.80

hard_negatives_idx = np.where(
    (~same_incident)
    &
    (
        similarities
        >
        HIGH_SIMILARITY
    )
)[0]

print(
    "Hard negatives:",
    len(hard_negatives_idx)
)

for idx in hard_negatives_idx[:10]:

    i = rows[idx]
    j = cols[idx]

    print("=" * 80)

    print(
        "Incident A:",
        pdf.iloc[i]["incident_id"]
    )

    print(
        "Ticket A:",
        pdf.iloc[i]["title"]
    )

    print()

    print(
        "Incident B:",
        pdf.iloc[j]["incident_id"]
    )

    print(
        "Ticket B:",
        pdf.iloc[j]["title"]
    )

    print(
        "\nSimilarity:",
        round(
            float(similarities[idx]),
            4
        )
    )

In [ ]:
# ============================================================
# 11. Hard positives
#
# Son pares que pertenecen al MISMO incidente pero presentan una similitud relativamente baja.
#
# Son candidatos potenciales a falsos negativos.
# ============================================================

LOW_SIMILARITY = 0.50

hard_positives_idx = np.where(
    same_incident
    &
    (
        similarities
        <
        LOW_SIMILARITY
    )
)[0]

print(
    "Hard positives:",
    len(hard_positives_idx)
)

for idx in hard_positives_idx[:10]:

    i = rows[idx]
    j = cols[idx]

    print("=" * 80)

    print(
        "Incident:",
        pdf.iloc[i]["incident_id"]
    )

    print(
        "Ticket A:",
        pdf.iloc[i]["title"]
    )

    print(
        "Ticket B:",
        pdf.iloc[j]["title"]
    )

    print(
        "\nSimilarity:",
        round(
            float(similarities[idx]),
            4
        )
    )

In [ ]:
# ============================================================
# 12. Evaluación experimental de thresholds
#
# Esta prueba NO determina los parámetros de DBSCAN.
#
# Su objetivo es entender hasta qué punto una simple regla:
#
#     similarity >= threshold
#
# sería capaz de discriminar incidentes iguales/diferentes.
#
# Sirve principalmente para demostrar el nivel de separabilidad de los embeddings.
# ============================================================

thresholds = np.arange(
    0.40,
    0.91,
    0.05
)

threshold_results = []

y_true = (
    pairs[
        "same_incident"
    ]
    .astype(bool)
)

for threshold in thresholds:

    y_pred = (
        pairs["similarity"]
        >= threshold
    )

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    threshold_results.append({
        "threshold":
            float(threshold),

        "precision":
            float(precision),

        "recall":
            float(recall),

        "f1":
            float(f1)
    })


threshold_results_pdf = (
    pd.DataFrame(
        threshold_results
    )
)

display(
    threshold_results_pdf
)

In [ ]:
# ============================================================
# 13. Mejor threshold según F1
#
# Se realiza adicionalmente un análisis de distintos umbrales de similitud coseno para estudiar la capacidad discriminativa de los embeddings. El umbral con mayor F1 se utiliza únicamente como referencia descriptiva y no como criterio de agrupamiento final, ya que la formación de clusters se realizó mediante DBSCAN.
# ============================================================

best_threshold = (
    threshold_results_pdf
    .sort_values(
        "f1",
        ascending=False
    )
    .iloc[0]
)

print(
    "Best experimental threshold:"
)

print(
    "Threshold:",
    round(
        best_threshold["threshold"],
        3
    )
)

print(
    "Precision:",
    round(
        best_threshold["precision"],
        4
    )
)

print(
    "Recall:",
    round(
        best_threshold["recall"],
        4
    )
)

print(
    "F1:",
    round(
        best_threshold["f1"],
        4
    )
)

In [1]:
# ============================================================
# 14. Relación entre threshold y métricas

# La idea es comprobar el efecto de ser más o menos estricto al decidir si dos tickets son similares:
#   - Si el threshold es bajo entonces más parejas son consideradas similares.
#   - Si el threshold es alto, menos parejas son consideradas similares
# ============================================================

plt.figure(
    figsize=(10, 6)
)

plt.plot(
    threshold_results_pdf[
        "threshold"
    ],
    threshold_results_pdf[
        "precision"
    ],
    marker="o",
    label="Precision"
)

plt.plot(
    threshold_results_pdf[
        "threshold"
    ],
    threshold_results_pdf[
        "recall"
    ],
    marker="o",
    label="Recall"
)

plt.plot(
    threshold_results_pdf[
        "threshold"
    ],
    threshold_results_pdf[
        "f1"
    ],
    marker="o",
    label="F1"
)

plt.xlabel(
    "Cosine similarity threshold"
)

plt.ylabel(
    "Score"
)

plt.title(
    "Pairwise classification by embedding similarity"
)

plt.legend()

plt.show()

NameError: name 'plt' is not defined

In [ ]:
# ============================================================
# 15. Matriz de confusión
# La matriz de confusión sirve para ver, para un threshold concreto, qué pares de tickets estamos clasificando correctamente o incorrectamente como relacionados.
# Utilizando el umbral de similitud con mejor F1 para analizar los aciertos y errores en la identificación de pares de tickets pertenecientes al mismo incidente. Este análisis se utiliza como evaluación descriptiva de los embeddings y no como mecanismo final de clustering.
# ============================================================

BEST_THRESHOLD = float(
    best_threshold[
        "threshold"
    ]
)

y_pred_best = (
    pairs["similarity"]
    >= BEST_THRESHOLD
)

cm = confusion_matrix(
    y_true,
    y_pred_best
)

print(
    f"Threshold: {BEST_THRESHOLD:.3f}"
)

print(
    "Confusion matrix:"
)

print(cm)

In [ ]:
# 16. Resumen del experimento
#
# En esta parte se genera un resumen de la evaluación realizada sobre los embeddings, recopilando las principales métricas y resultados obtenidos.
#
# Esto permite valorar de forma conjunta la capacidad de los embeddings para diferenciar tickets relacionados y no relacionados.

print(
    "Embedding evaluation completed."
)

print(
    "--------------------------------"
)

print(
    "Tickets evaluated:",
    len(pdf)
)

print(
    "Pairwise comparisons:",
    len(pairs)
)

print(
    "Mean similarity - same incident:",
    round(mean_same, 4)
)

print(
    "Mean similarity - different incident:",
    round(mean_different, 4)
)

print(
    "Hard negatives (> 0.80):",
    len(hard_negatives_idx)
)

print(
    "Hard positives (< 0.50):",
    len(hard_positives_idx)
)

print(
    "Best descriptive threshold:",
    round(
        BEST_THRESHOLD,
        3
    )
)

print(
    "Best threshold F1:",
    round(
        best_threshold["f1"],
        4
    )
)